In [61]:
# IMPORT THE REQUIRED LIBRARIES FOR DATA
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# READ THE TRAINING AND TEST DATASETS FROM CSV FILES
train_DF = pd.read_csv("trained_dataset.csv")
test_DF = pd.read_csv("test_dataset.csv")

# DISPLAY ALL ROWS AND COLUMNS WITHOUT TRUNCATION FOR FULL INSPECTION
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# DISPLAY THE DIMENSIONS OF BOTH DATASETS
print(f":> TRAINING DATASET SHAPE: {train_DF.shape}")
print(f":> TEST DATASET SHAPE: {test_DF.shape}")

:> TRAINING DATASET SHAPE: (1460, 81)
:> TEST DATASET SHAPE: (1459, 80)


In [62]:
# SEPARATE FEATURES AND TARGET VARIABLE FROM TRAINING DATA
X_train = train_DF.drop(columns='SalePrice')
y_train = train_DF['SalePrice']

# CREATE A COPY OF TEST DATASET FOR FEATURES
X_test = test_DF.copy()

# CALCULATE MISSING VALUES COUNT FOR EACH COLUMN
Null_Val = X_train.isnull().sum()

# IDENTIFY NUMERICAL AND CATEGORICAL COLUMNS
Numric_Cols = X_train.select_dtypes(include=['int64', 'float64']).columns
Cate_Cols = X_train.select_dtypes(include=['object']).columns

# FILTER ONLY THOSE COLUMNS THAT HAVE MISSING VALUES
Drp_Numric_Cols = [i for i in Numric_Cols if Null_Val[i] > 0]
Drp_Cate_Cols = [i for i in Cate_Cols if Null_Val[i] > 0]

C:\Users\Admin\AppData\Local\Temp\ipykernel_15988\1890847959.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  Cate_Cols = X_train.select_dtypes(include=['object']).columns


In [63]:
# DISPLAY COLUMNS WITH MISSING VALUES BY DATA TYPE
print(f":> NUMERICAL COLUMNS CONTAINING MISSING VALUES: {Drp_Numric_Cols}")
print(f":> CATEGORICAL COLUMNS CONTAINING MISSING VALUES: {Drp_Cate_Cols}")

display(train_DF[Drp_Numric_Cols].isnull().sum())       # DISPLAY MISSING VALUE COUNTS FOR NUMERICAL COLUMNS
display(train_DF[Drp_Cate_Cols].isnull().sum())         # DISPLAY MISSING VALUE COUNTS FOR CATEGORICAL COLUMNS

:> NUMERICAL COLUMNS CONTAINING MISSING VALUES: ['LotFrontage', 'MasVnrArea', 'GarageYrBlt']
:> CATEGORICAL COLUMNS CONTAINING MISSING VALUES: ['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Electrical', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature']


LotFrontage    259
MasVnrArea       8
GarageYrBlt     81
dtype: int64

Alley           1369
MasVnrType       872
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64

In [ ]:
# For_Mean_Cols = [col for col in Drp_Numric_Cols if X_train[col].nunique() > 20]                 # USE MEAN FOR CONTINUOUS NUMERICAL FEATURES
# For_Median_Cols = [col for col in Drp_Numric_Cols if X_train[col].nunique() <= 20]              # USE MEDIAN FOR SKEWED / SMALL NUMERICAL FEATURES
# For_Mode_Cols = [col for col in Drp_Cate_Cols if X_train[col].isnull().mean() < 0.5]            # USE MODE FOR NORMAL CATEGORICAL FEATURES
# For_Missing_Cols = [col for col in Drp_Cate_Cols if X_train[col].isnull().mean() >= 0.5]        # USE "missing" IF TOO MANY NULL VALUES

# # DISPLAY RESULTS
# print(f":> MEAN COLS: {For_Mean_Cols}")
# print(f":> MEDIAN COLS: {For_Median_Cols}")
# print(f":> MODE COLS: {For_Mode_Cols}")
# print(f":> MISSING COLS: {For_Missing_Cols}")

# # NOTE: IT'S ADVANCED METHODS TO CALCULATE DIFFERENT CSV FILE VALUEs (AUTOMATIC SELECTION)

:> MEAN COLS: ['LotFrontage', 'MasVnrArea', 'GarageYrBlt']
:> MEDIAN COLS: []
:> MODE COLS: ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Electrical', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']
:> MISSING COLS: ['Alley', 'MasVnrType', 'PoolQC', 'Fence', 'MiscFeature']


In [ ]:
# DEFINE COLUMNS TO BE IMPUTED WITH MEAN STRATEGY
For_Mean_Cols = ["LotFrontage"]

# DEFINE COLUMNS TO BE IMPUTED WITH MEDIAN STRATEGY
For_Median_Cols = ["MasVnrArea", "GarageYrBlt"]

# DEFINE COLUMNS TO BE IMPUTED WITH MODE (MOST FREQUENT) STRATEGY
For_Mode_Cols = [
    "Alley",
    "MasVnrType",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "Electrical",
    "FireplaceQu",
]

# DEFINE COLUMNS TO BE IMPUTED WITH CONSTANT VALUE "missing"
For_Missing_Cols = [
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "PoolQC",
    "Fence",
    "MiscFeature",
]


# CREATE PIPELINES FOR DIFFERENT IMPUTATION STRATEGIES
Mean_Impute = Pipeline(steps=[("imputer", SimpleImputer(strategy="mean"))])
Median_Impute = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
Mode_Impute = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent"))])
Missing_Impute = Pipeline(steps=[("imputer", SimpleImputer(strategy="constant", fill_value="missing"))])

# CREATE COLUMN TRANSFORMER TO APPLY DIFFERENT STRATEGIES TO SPECIFIC COLUMNS
preprocessor = ColumnTransformer(transformers=[
    ('mean_imputer', Mean_Impute, For_Mean_Cols),
    ('median_imputer', Median_Impute, For_Median_Cols),
    ('mode_imputer', Mode_Impute, For_Mode_Cols),
    ('missing_imputer', Missing_Impute, For_Missing_Cols),
])

In [66]:
# FIT THE PREPROCESSOR ON BOTH TRAINING AND TEST DATA
preprocessor.fit(X_train)
# preprocessor.fit(X_test)

# EXTRACT THE CALCULATED STATISTICS FROM EACH IMPUTER
Mean_Values = preprocessor.named_transformers_["mean_imputer"].named_steps["imputer"].statistics_
Median_Values = preprocessor.named_transformers_["median_imputer"].named_steps["imputer"].statistics_
Mode_Values = preprocessor.named_transformers_["mode_imputer"].named_steps["imputer"].statistics_

# TRANSFORM BOTH TRAINING AND TEST DATASETS
Clean_X_train = preprocessor.transform(X_train)
Clean_X_test = preprocessor.transform(X_test)

# DISPLAY THE TRANSFORMED TRAINING DATA
print(":> TRAINING DATA AFTER IMPUTATION (ALL STRATEGIES APPLIED) >>")
display(Clean_X_train)

# DISPLAY THE TRANSFORMED TEST DATA
print("\n:> TEST DATA AFTER IMPUTATION (ALL STRATEGIES APPLIED) >>")
display(Clean_X_test)

AttributeError: 'SimpleImputer' object has no attribute 'statistics_'

In [ ]:
# DISPLAY THE TRANSFORMERS CONFIGURATION FOR VERIFICATION
print(preprocessor.transformers_)

[('mean_imputer', Pipeline(steps=[('imputer', SimpleImputer())]), ['LotFrontage']), ('median_imputer', Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]), ['MasVnrArea', 'GarageYrBlt']), ('mode_imputer', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent'))]), ['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Electrical', 'FireplaceQu']), ('missing_imputer', Pipeline(steps=[('imputer',
                 SimpleImputer(fill_value='missing', strategy='constant'))]), ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature']), ('remainder', 'drop', ['Id', 'MSSubClass', 'MSZoning', 'LotArea', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation

In [ ]:
# CONVERT TRANSFORMED ARRAYS BACK TO DATAFRAMES FOR BETTER VISUALIZATION
Complete_X_train = pd.DataFrame(Clean_X_train, columns=For_Mean_Cols + For_Median_Cols + For_Mode_Cols + For_Missing_Cols)
Complete_X_test = pd.DataFrame(Clean_X_test, columns=For_Mean_Cols + For_Median_Cols + For_Mode_Cols + For_Missing_Cols)

In [ ]:
# DISPLAY THE COMPLETE CLEANED TRAINING DATAFRAME
display(Complete_X_train)

,LotFrontage,MasVnrArea,GarageYrBlt,Alley,MasVnrType,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,Electrical,FireplaceQu,GarageType,GarageFinish,GarageQual,GarageCond,PoolQC,Fence,MiscFeature
0,65.0,196.0,2003.0,Grvl,BrkFace,Gd,TA,No,GLQ,Unf,SBrkr,Gd,Attchd,RFn,TA,TA,missing,missing,missing
1,80.0,0.0,1976.0,Grvl,BrkFace,Gd,TA,Gd,ALQ,Unf,SBrkr,TA,Attchd,RFn,TA,TA,missing,missing,missing
2,68.0,162.0,2001.0,Grvl,BrkFace,Gd,TA,Mn,GLQ,Unf,SBrkr,TA,Attchd,RFn,TA,TA,missing,missing,missing
3,60.0,0.0,1998.0,Grvl,BrkFace,TA,Gd,No,ALQ,Unf,SBrkr,Gd,Detchd,Unf,TA,TA,missing,missing,missing
4,84.0,350.0,2000.0,Grvl,BrkFace,Gd,TA,Av,GLQ,Unf,SBrkr,TA,Attchd,RFn,TA,TA,missing,missing,missing
5,85.0,0.0,1993.0,Grvl,BrkFace,Gd,TA,No,GLQ,Unf,SBrkr,Gd,Attchd,Unf,TA,TA,missing,MnPrv,Shed
6,75.0,186.0,2004.0,Grvl,Stone,Ex,TA,Av,GLQ,Unf,SBrkr,Gd,Attchd,RFn,TA,TA,missing,missing,missing
7,70.049958,240.0,1973.0,Grvl,Stone,Gd,TA,Mn,ALQ,BLQ,SBrkr,TA,Attchd,RFn,TA,TA,missing,missing,Shed
8,51.0,0.0,1931.0,Grvl,BrkFace,TA,TA,No,Unf,Unf,FuseF,TA,Detchd,Unf,Fa,TA,missing,missing,missing
9,50.0,0.0,1939.0,Grvl,BrkFace,TA,TA,No,GLQ,Unf,SBrkr,TA,Attchd,RFn,Gd,TA,missing,missing,missing


In [ ]:
# DISPLAY THE COMPLETE CLEANED TEST DATAFRAME
display(Complete_X_test)

,LotFrontage,MasVnrArea,GarageYrBlt,Alley,MasVnrType,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,Electrical,FireplaceQu,GarageType,GarageFinish,GarageQual,GarageCond,PoolQC,Fence,MiscFeature
0,80.0,0.0,1961.0,Grvl,BrkFace,TA,TA,No,Rec,LwQ,SBrkr,Gd,Attchd,Unf,TA,TA,missing,MnPrv,missing
1,81.0,108.0,1958.0,Grvl,BrkFace,TA,TA,No,ALQ,Unf,SBrkr,Gd,Attchd,Unf,TA,TA,missing,missing,Gar2
2,74.0,0.0,1997.0,Grvl,BrkFace,Gd,TA,No,GLQ,Unf,SBrkr,TA,Attchd,Fin,TA,TA,missing,MnPrv,missing
3,78.0,20.0,1998.0,Grvl,BrkFace,TA,TA,No,GLQ,Unf,SBrkr,Gd,Attchd,Fin,TA,TA,missing,missing,missing
4,43.0,0.0,1992.0,Grvl,BrkFace,Gd,TA,No,ALQ,Unf,SBrkr,Gd,Attchd,RFn,TA,TA,missing,missing,missing
5,75.0,0.0,1993.0,Grvl,BrkFace,Gd,TA,No,Unf,Unf,SBrkr,TA,Attchd,Fin,TA,TA,missing,missing,missing
6,70.049958,0.0,1992.0,Grvl,BrkFace,Gd,TA,No,ALQ,Unf,SBrkr,Gd,Attchd,Fin,TA,TA,missing,GdPrv,Shed
7,63.0,0.0,1998.0,Grvl,BrkFace,Gd,TA,No,Unf,Unf,SBrkr,Gd,Attchd,Fin,TA,TA,missing,missing,missing
8,85.0,0.0,1990.0,Grvl,BrkFace,Gd,TA,Gd,GLQ,Unf,SBrkr,Po,Attchd,Unf,TA,TA,missing,missing,missing
9,70.0,0.0,1970.0,Grvl,BrkFace,TA,TA,No,ALQ,Rec,SBrkr,Gd,Attchd,Fin,TA,TA,missing,MnPrv,missing


In [ ]:
# CALCULATE TOTAL MISSING VALUES IN THE CLEANED DATASETS
Null_Miss1 = Complete_X_train.isnull().sum().sum()
Null_Miss2 = Complete_X_train.isnull().sum().sum()

# DISPLAY FINAL SHAPES OF CLEANED DATASETS
print(f":> CLEANED TRAINING DATASET SHAPE: {Complete_X_train.shape}")
print(f":> CLEANED TEST DATASET SHAPE: {Complete_X_test.shape}")

# DISPLAY FINAL MISSING VALUE COUNTS (SHOULD BE ZERO)
print(f"\n:> TOTAL MISSING VALUES IN TRAINING DATASET (FINAL): {Null_Miss1}")
print(f":> TOTAL MISSING VALUES IN TEST DATASET (FINAL): {Null_Miss2}")

:> CLEANED TRAINING DATASET SHAPE: (1460, 19)
:> CLEANED TEST DATASET SHAPE: (1459, 19)

:> TOTAL MISSING VALUES IN TRAINING DATASET (FINAL): 0
:> TOTAL MISSING VALUES IN TEST DATASET (FINAL): 0
